# GCSI Bulk History

## From the Datasets page to a clean daily panel, 2015 onwards

The API serves a rolling window of recent data. For long history, download the bulk files: one ZIP per commodity
holding its full hourly index. This notebook:

1. Finds the GCSI hourly dataset and downloads one ZIP per ticker through the Datasets API
2. Reads the ZIP into a long hourly frame
3. Runs the checks worth doing before any research: coverage, sparsity, and index-type consistency
4. Builds a **daily panel** at three levels (topic, driver, all topics), with correctly pooled statistics
5. Saves it as Parquet for the topic and backtesting notebooks

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment scores are model outputs describing how news was judged at publication; they are not price forecasts.

> **Backfill timing:** for the historical (backfill) period, the time each headline was received was reconstructed.
> Treat point-in-time accuracy before the live period as approximate, and leave a lag in any research.

In [ ]:
%pip install requests pandas numpy matplotlib pyarrow

## Step 1: Configuration

In [ ]:
import getpass
import os
import random
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests


def load_env_file() -> None:
    """Read KEY=VALUE pairs from the nearest .env file without overriding variables already set."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / '.env'
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"'))
            return


def get_api_key() -> str:
    load_env_file()
    key = os.environ.get('API_KEY') or os.environ.get('PERMUTABLE_API_KEY')
    if not key:
        key = getpass.getpass('Enter your Permutable AI API key: ')
    return key


TICKERS = ['BZ_COM', 'CL_COM']
DOWNLOAD = True                       # False = use ZIPs already in ZIP_DIR (e.g. downloaded by hand)
ZIP_DIR = Path('data/gcsi_zips')
OUTPUT_DIR = Path('data/gcsi_daily')
INDEX_TYPES = ['COMBINED', 'EXPLICIT', 'RELATED_ENTITY', 'MACROECONOMIC']
SENTIMENT_TYPES = ['short_term', 'medium_term', 'long_term']

DATASET_CATEGORY = 'headline_gcsi_hour'   # placeholder: GCSI category on GET /v1/datasets/categories
BUNDLE_DIMENSION = 'ticker'               # placeholder: top-level bundle dimension of the GCSI dataset

BASE_URL = os.environ.get('BASE_URL', 'https://copilot-api.permutable.ai/v1')

# GCSI routes. The ticker and topic are query presets, not part of the path.
INDEX_LIVE_PATH = '/headlines/index/gcsi/live'
INDEX_HISTORICAL_PATH = '/headlines/index/gcsi/historical'
FEED_LIVE_PATH = '/headlines/feed/live/gcsi'
FEED_HISTORICAL_PATH = '/headlines/feed/historical/gcsi'
PAGE_LIMIT = 1000
MAX_LOOKBACK_DAYS = 90

API_KEY = get_api_key() if DOWNLOAD else None
ZIP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 2: Download through the Datasets API

The flow is: find the latest active hourly dataset in the category, list its per-ticker bundles, then ask for a
short-lived download link for each bundle. The link is a signed URL, so treat it like a password: it is never
printed or logged here. Existing files are not downloaded again.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({'x-api-key': API_KEY, 'accept': 'application/json'})


def api_get(path: str, params: dict | None = None, max_retries: int = 5):
    """GET a Permutable API path, backing off exponentially (with jitter) on 429 and 5xx responses."""
    params = {k: v for k, v in (params or {}).items() if v is not None}
    delay = 1.0
    for attempt in range(max_retries + 1):
        response = SESSION.get(BASE_URL + path, params=params, timeout=60)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt == max_retries:
                response.raise_for_status()
            time.sleep(delay + random.uniform(0, delay / 2))
            delay *= 2
            continue
        if response.status_code in (401, 403):
            raise PermissionError(
                f'{response.status_code} on {path}: check the API key, and that your licence includes GCSI.'
            )
        response.raise_for_status()
        return response.json()


def _records(body) -> list[dict]:
    """Live endpoints return a list; historical endpoints return {data, has_more, next_token}."""
    return body['data'] if isinstance(body, dict) else body


def _stamp(records: list[dict], **dims) -> list[dict]:
    """Responses can omit the dimensions you asked for, so write them back onto each record."""
    for record in records:
        for key, value in dims.items():
            if value is not None:
                record.setdefault(key, value)
    return records


def _paginate(path: str, params: dict) -> list[dict]:
    params = {**params, 'limit': PAGE_LIMIT}
    rows: list[dict] = []
    while True:
        body = api_get(path, params)
        rows.extend(_records(body))
        if not (isinstance(body, dict) and body.get('has_more')):
            return rows
        params['next_token'] = body['next_token']


TOPIC_SEPARATOR = '::'  # Sector::L1::L2, the form the API returns


def normalise_topic(labels: pd.Series) -> pd.Series:
    """Write each label as sector, driver and topic joined by TOPIC_SEPARATOR, whatever separator the source used."""
    return labels.astype('string').str.replace(r'\s*(?:::|-)\s*', TOPIC_SEPARATOR, regex=True)


def latest_dataset(category: str) -> dict:
    datasets = api_get('/datasets', {'category': category, 'page_size': 50})
    datasets = datasets.get('data', datasets) if isinstance(datasets, dict) else datasets
    active = [d for d in datasets if d.get('status') == 'ACTIVE' and d.get('granularity') == 'HOUR']
    if not active:
        raise LookupError(f'No active hourly dataset in category {category!r}; check GET /datasets/categories')
    return max(active, key=lambda d: d.get('data_end_date') or '')


def bundles(dataset_id: str) -> dict[str, str]:
    """Map each top-level bundle (one per ticker) to its bundle-file id."""
    children = api_get(f'/datasets/{dataset_id}/tree-children')
    return {
        child['dimension_value']: child['bundle_file']['id']
        for child in children
        if child.get('dimension_key') == BUNDLE_DIMENSION and child.get('bundle_file')
    }


def download_bundle(dataset_id: str, bundle_id: str, target: Path) -> Path:
    link = api_get(f'/datasets/{dataset_id}/versions/{bundle_id}/download', {'expires_in': 900})['url']
    with requests.get(link, stream=True, timeout=600) as response:   # signed URL: no API key header
        response.raise_for_status()
        partial = target.with_suffix('.part')
        with partial.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=1 << 20):
                handle.write(chunk)
    partial.replace(target)
    return target


if DOWNLOAD:
    dataset = latest_dataset(DATASET_CATEGORY)
    print(f"Dataset: {dataset['name']} | data {dataset['data_start_date'][:10]} → {dataset['data_end_date'][:10]}")
    available = bundles(dataset['id'])
    for ticker in TICKERS:
        target = ZIP_DIR / f'{ticker}.zip'
        if target.exists():
            print(f'{ticker}: already downloaded')
        elif ticker not in available:
            print(f'{ticker}: not in this dataset (or not in your licence)')
        else:
            download_bundle(dataset['id'], available[ticker], target)
            print(f'{ticker}: {target.stat().st_size / 1e6:,.0f} MB')
else:
    print('Using local files:', sorted(p.name for p in ZIP_DIR.glob('*.zip')))

## Step 3: Read the hourly files

`read_index_zip` accepts both folder layouts (`index_type=…/sentiment_type=…/` or topic folders) and adds any
missing value column as empty, so the rest of the notebook works either way.

In [ ]:
INDEX_VALUES = ['headline_count', 'sentiment_sum', 'sentiment_std']


def read_index_zip(
    path: Path, index_types: list[str] | None = None, sentiment_types: list[str] | None = None
) -> pd.DataFrame:
    """Read one ticker's bulk hourly-index ZIP into a long frame.

    Handles key=value folders (index_type=COMBINED/sentiment_type=short_term/part.csv) and plain topic
    folders. Dimensions found only in the folder path are added as columns, value columns missing from
    the file are added as NaN, and topic labels are written in the `::` form.
    """
    ticker = Path(path).stem
    frames = []
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith('.csv'):
                continue
            folders = name.split('/')[:-1]
            dims = dict(part.split('=', 1) for part in folders if '=' in part)
            plain = [part for part in folders if '=' not in part and part != ticker]
            if index_types and dims.get('index_type', index_types[0]) not in index_types:
                continue
            if sentiment_types and dims.get('sentiment_type', sentiment_types[0]) not in sentiment_types:
                continue
            with archive.open(name) as handle:
                frame = pd.read_csv(handle)
            for key, value in dims.items():
                if key not in frame:
                    frame[key] = value
            frame = frame.rename(columns={'publication_time': 'extraction_time', 'topic': 'topic_name'})
            if 'topic_name' not in frame and plain:
                frame['topic_name'] = plain[-1]
            frames.append(frame)
    if not frames:
        raise ValueError(f'{path}: no CSV files matched the requested index and sentiment types')
    frame = pd.concat(frames, ignore_index=True)
    if 'ticker' not in frame:
        frame['ticker'] = ticker
    for column in INDEX_VALUES:
        if column not in frame:
            frame[column] = np.nan
    frame['extraction_time'] = pd.to_datetime(frame['extraction_time'], utc=True)  # files carry UTC times
    frame['topic_name'] = normalise_topic(frame['topic_name'])
    if index_types:
        frame = frame[frame['index_type'].isin(index_types)]
    if sentiment_types:
        frame = frame[frame['sentiment_type'].isin(sentiment_types)]
    return frame.reset_index(drop=True)


hourly = {}
for ticker in TICKERS:
    path = ZIP_DIR / f'{ticker}.zip'
    if path.exists():
        hourly[ticker] = read_index_zip(path, INDEX_TYPES, SENTIMENT_TYPES)
        frame = hourly[ticker]
        print(f"{ticker}: {len(frame):>10,} rows | {frame['topic_name'].nunique()} topics | "
              f"{frame['extraction_time'].min():%Y-%m-%d} → {frame['extraction_time'].max():%Y-%m-%d}")

## Step 4: Checks before research

- **Coverage:** active hours per year. Early years carry less news, and a z-score across a step change in volume
  will mislead.
- **Sparsity:** the share of hours with any news, and the share of bars built from a single headline. For those
  bars `sentiment_std` is 0 by definition.
- **Index types:** each headline is assigned to one match type (priority `EXPLICIT`, then `RELATED_ENTITY`,
  then `MACROECONOMIC`), so `COMBINED` should equal the sum of the other three. Adding all four together would
  count every headline twice.

In [ ]:
for ticker, frame in hourly.items():
    base = frame[(frame['index_type'] == 'COMBINED') & (frame['sentiment_type'] == 'short_term')]
    hours = base.groupby(base['extraction_time'].dt.year)['extraction_time'].nunique()
    links = base.groupby(base['extraction_time'].dt.year)['headline_count'].sum()
    print(f'\n{ticker}: active hours and topic-links per year (COMBINED, short_term)')
    display(pd.DataFrame({'active_hours': hours, 'share_of_hours': (hours / (365.25 * 24)).round(2),
                          'topic_links': links}).T)
    print(f"Bars from a single headline: {(base['headline_count'] == 1).mean():.0%}")

    per_type = (
        frame[frame['sentiment_type'] == 'short_term']
        .groupby('index_type')['headline_count'].sum()
    )
    parts = per_type.drop('COMBINED', errors='ignore').sum()
    if 'COMBINED' in per_type and parts:
        print(f"COMBINED topic-links / sum of the other index types: {per_type['COMBINED'] / parts:.2f}")

## Step 5: Daily panel at three levels

Hourly bars are grouped by the UTC day the headlines were received. Each day is then pooled at three levels:

| `level` | `name` example | Meaning |
|---|---|---|
| `topic` | `Energy::Supply::Outages & Disruptions` | One L2 topic |
| `driver` | `Energy::Supply` | All topics under one L1 driver |
| `all` | `ALL` | Every topic of the commodity |

Above the topic level, counts are **topic-links**, because one headline can sit in several topics. Standard
deviations are pooled exactly (`pool`), never averaged.

In [ ]:
def pool(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Combine index rows into larger buckets (hours → days, topics → drivers, ...).

    Counts and sums add. The sample standard deviation is pooled exactly: each row's sum of squares is
    rebuilt as (n - 1)·std² + n·mean², summed, and turned back into a sample standard deviation.
    Averaging hourly standard deviations would be wrong.
    """
    n = frame['headline_count'].astype(float)
    mean = frame['sentiment_sum'] / n
    sumsq = (n - 1).clip(lower=0) * frame['sentiment_std'].fillna(0.0) ** 2 + n * mean**2
    out = (
        frame.assign(_sumsq=sumsq)
        .groupby(keys, as_index=False, observed=True)[['headline_count', 'sentiment_sum', '_sumsq']]
        .sum()
    )
    n = out['headline_count'].astype(float)
    out['sentiment_avg'] = out['sentiment_sum'] / n
    variance = (out['_sumsq'] - n * out['sentiment_avg'] ** 2) / (n - 1)
    out['sentiment_std'] = np.sqrt(variance.where(n > 1, 0.0).clip(lower=0.0))
    return out.drop(columns='_sumsq')


def receipt_date(extraction_time: pd.Series) -> pd.Series:
    """UTC date the headlines were received. A bar stamped 00:00 covers 23:00–24:00 of the previous day."""
    return (extraction_time - pd.Timedelta(hours=1)).dt.floor('D').dt.tz_localize(None)


def daily_panel(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.assign(date=receipt_date(frame['extraction_time']))
    keys = ['date', 'index_type', 'sentiment_type']
    topic = pool(frame, keys + ['topic_name']).rename(columns={'topic_name': 'name'}).assign(level='topic')
    frame['driver'] = frame['topic_name'].str.rsplit(TOPIC_SEPARATOR, n=1).str[0]   # Sector::L1
    driver = pool(frame, keys + ['driver']).rename(columns={'driver': 'name'}).assign(level='driver')
    overall = pool(frame, keys).assign(name='ALL', level='all')
    return pd.concat([topic, driver, overall], ignore_index=True)


panels = {}
for ticker, frame in hourly.items():
    panel = daily_panel(frame).assign(ticker=ticker)
    panel.to_parquet(OUTPUT_DIR / f'{ticker}_daily.parquet', index=False)
    panels[ticker] = panel
    print(f"{ticker}: {len(panel):,} daily rows → {OUTPUT_DIR / f'{ticker}_daily.parquet'}")

display(panels[TICKERS[0]].query("level == 'all' and sentiment_type == 'short_term'").tail())

## Step 6: Long-run view

A 30-day weighted rolling average (`Σ sentiment_sum / Σ headline_count` over a 30-calendar-day window) for each
horizon, with daily volume underneath. Days without news add nothing to either sum, so they are handled
correctly without zero-filling.

In [ ]:
for ticker, panel in panels.items():
    sub = panel[(panel['level'] == 'all') & (panel['index_type'] == 'COMBINED')]
    fig, (ax, ax_vol) = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    for sentiment_type, group in sub.groupby('sentiment_type'):
        rolled = group.set_index('date')[['sentiment_sum', 'headline_count']].rolling('30D').sum()
        ax.plot(rolled.index, rolled['sentiment_sum'] / rolled['headline_count'], lw=1.2, label=sentiment_type)
    ax.axhline(0, color='grey', lw=0.8, ls=':')
    ax.set_title(f'{ticker} | COMBINED | 30-day weighted average sentiment by horizon')
    ax.legend()
    volume = sub[sub['sentiment_type'] == 'short_term'].set_index('date')['headline_count']
    ax_vol.fill_between(volume.index, volume.rolling('30D').mean(), color='#7f8c8d', step='mid')
    ax_vol.set_ylabel('Topic-links/day\n(30d mean)')
    plt.tight_layout()
    plt.show()